# DB Structure Inspector

MySQL DB table list, column metadata, and sample rows can be inspected from this notebook.

Run the cells from top to bottom. DB login values are set directly in the connection settings cell.

In [41]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, datetime
from decimal import Decimal
from typing import Any, Iterable, Sequence

# pandas가 있으면 노트북에서 DataFrame 형태로 보기 좋게 출력합니다.
# 없더라도 아래의 ASCII 테이블 출력 함수로 동작할 수 있게 처리합니다.
try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

@dataclass
class DbConfig:
    host: str = "127.0.0.1"
    port: int = 3306
    user: str = "root"
    password: str = ""
    name: str = "farmstom"


## 1. Connection Settings

Change these values if you need a different DB host, user, database, or table.

In [42]:
# 기존 updated/ 코드에서 사용하던 DB 접속 정보를 그대로 직접 입력합니다.
cfg = DbConfig(
    host="211.195.9.227",
    port=3306,
    user="root",
    password="theimc#10!",
    name="farmstom",
)

TABLE = "data_silla_enc"
WHERE = "iot_data_idx = 97"
ORDER_BY = "reg_date"
SAMPLE_SIZE = 5
MAX_WIDTH = 36

print(f"[DB] host={cfg.host}, port={cfg.port}, user={cfg.user}, database={cfg.name}")
print("[source] hardcoded connection settings; password is hidden")
print(f"[TABLE] {TABLE}")


[DB] host=211.195.9.227, port=3306, user=root, database=farmstom
[source] hardcoded connection settings; password is hidden
[TABLE] data_silla_enc


## 2. Helper Functions

In [43]:
def quote_identifier(name: str) -> str:
    if not name or "\x00" in name:
        raise ValueError(f"invalid SQL identifier: {name!r}")
    return "`" + name.replace("`", "``") + "`"


def connect(cfg: DbConfig):
    # DictCursor를 사용하면 조회 결과가 {컬럼명: 값} 형태로 나와 표로 만들기 쉽습니다.
    try:
        import pymysql
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "pymysql is required to inspect MySQL. Install project requirements "
            "or run this notebook in the GEAS Python environment."
        ) from exc

    try:
        conn = pymysql.connect(
            host=cfg.host,
            port=cfg.port,
            user=cfg.user,
            password=cfg.password,
            database=cfg.name,
            charset="utf8mb4",
            autocommit=True,
            cursorclass=pymysql.cursors.DictCursor,
        )
    except pymysql.err.OperationalError as exc:
        raise SystemExit(
            "MySQL connection failed.\n"
            f"  host={cfg.host}, port={cfg.port}, user={cfg.user}, database={cfg.name}\n"
            "Check that the host and port point to the real MySQL server, "
            "and that the server allows connections from this PC.\n"
            f"Original error: {exc}"
        ) from exc

    # 시간대와 문자셋을 프로젝트 기준에 맞춥니다.
    with conn.cursor() as cur:
        cur.execute("SET time_zone = %s", ("+09:00",))
        cur.execute("SET NAMES utf8mb4")
    return conn


def stringify(value: Any) -> str:
    # 날짜, Decimal, bytes처럼 바로 출력하기 애매한 값을 문자열로 정리합니다.
    if value is None:
        return "NULL"
    if isinstance(value, (datetime, date)):
        return value.isoformat(sep=" ")
    if isinstance(value, Decimal):
        return format(value, "f")
    if isinstance(value, bytes):
        return value.hex()
    return str(value)


def shrink(text: str, width: int) -> str:
    text = " ".join(text.replace("\r", " ").replace("\n", " ").split())
    if len(text) <= width:
        return text
    if width <= 3:
        return text[:width]
    return text[: width - 3] + "..."


def print_table(rows: Iterable[dict[str, Any]], headers: Sequence[str], *, max_width: int = 36) -> None:
    # pandas가 없는 환경에서도 결과를 읽을 수 있도록 단순 ASCII 표를 만듭니다.
    materialized = [
        {key: shrink(stringify(row.get(key)), max_width) for key in headers}
        for row in rows
    ]
    widths = {
        key: max([len(key)] + [len(row[key]) for row in materialized])
        for key in headers
    }

    line = "+-" + "-+-".join("-" * widths[key] for key in headers) + "-+"
    print(line)
    print("| " + " | ".join(key.ljust(widths[key]) for key in headers) + " |")
    print(line)
    for row in materialized:
        print("| " + " | ".join(row[key].ljust(widths[key]) for key in headers) + " |")
    print(line)
    print(f"{len(materialized)} row(s)\n")


def show_table(rows: list[dict[str, Any]], columns: Sequence[str] | None = None) -> None:
    # 노트북에서는 pandas DataFrame으로, 일반 콘솔에서는 ASCII 표로 보여줍니다.
    if columns is not None:
        rows = [{key: row.get(key) for key in columns} for row in rows]
    if pd is not None:
        display(pd.DataFrame(rows))
    elif rows:
        print_table(rows, list(rows[0].keys()), max_width=MAX_WIDTH)
    else:
        print("No rows.")


def fetch_tables(conn, schema: str) -> list[dict[str, Any]]:
    # information_schema.TABLES에서 현재 DB의 테이블 목록과 대략적인 row 수를 가져옵니다.
    sql = """
        SELECT
            TABLE_NAME AS table_name,
            TABLE_TYPE AS table_type,
            ENGINE AS engine,
            TABLE_ROWS AS approx_rows,
            CREATE_TIME AS create_time,
            UPDATE_TIME AS update_time
        FROM information_schema.TABLES
        WHERE TABLE_SCHEMA = %s
        ORDER BY TABLE_NAME
    """
    with conn.cursor() as cur:
        cur.execute(sql, (schema,))
        return list(cur.fetchall())


def fetch_columns(conn, schema: str, table: str) -> list[dict[str, Any]]:
    # information_schema.COLUMNS에서 컬럼명, 타입, NULL 허용 여부, 키 정보를 가져옵니다.
    sql = """
        SELECT
            ORDINAL_POSITION AS pos,
            COLUMN_NAME AS column_name,
            COLUMN_TYPE AS column_type,
            IS_NULLABLE AS nullable,
            COLUMN_KEY AS column_key,
            COLUMN_DEFAULT AS default_value,
            EXTRA AS extra
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = %s
          AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION
    """
    with conn.cursor() as cur:
        cur.execute(sql, (schema, table))
        return list(cur.fetchall())


def fetch_sample_rows(conn, table: str, *, limit: int, order_by: str | None = None, where: str | None = None) -> list[dict[str, Any]]:
    # 실제 테이블에서 예시 행을 가져옵니다. where는 필요한 경우 필터 조건으로 사용합니다.
    order_sql = f" ORDER BY {quote_identifier(order_by)} DESC" if order_by else ""
    where_sql = f" WHERE {where}" if where else ""
    sql = f"SELECT * FROM {quote_identifier(table)}{where_sql}{order_sql} LIMIT %s"
    with conn.cursor() as cur:
        cur.execute(sql, (limit,))
        return list(cur.fetchall())


def inspect_table(conn, *, schema: str, table: str, sample_size: int = 5, order_by: str | None = "reg_date", where: str | None = None) -> None:
    # 한 테이블에 대해 컬럼 구조와 샘플 데이터를 연달아 출력하는 메인 확인 함수입니다.
    columns = fetch_columns(conn, schema, table)
    if not columns:
        print(f"Table '{table}' was not found in database '{schema}'.")
        return

    print(f"## Columns: {schema}.{table}")
    show_table(columns)

    # 지정한 정렬 컬럼이 없는 테이블에서는 정렬 없이 샘플만 가져옵니다.
    column_names = {row["column_name"] for row in columns}
    effective_order_by = order_by if order_by in column_names else None
    samples = fetch_sample_rows(
        conn,
        table,
        limit=max(sample_size, 0),
        order_by=effective_order_by,
        where=where,
    )

    print(f"## Sample Rows: {schema}.{table}")
    show_table(samples)


## 3. Connect and List Tables

In [44]:
# DB에 연결한 뒤 전체 테이블 목록을 먼저 확인합니다.
conn = connect(cfg)
tables = fetch_tables(conn, cfg.name)
show_table(tables)


,table_name,table_type,engine,approx_rows,create_time,update_time
0,account_book,BASE TABLE,InnoDB,0,2024-05-14 10:12:08,NaT
1,admin_auto_column_info,BASE TABLE,InnoDB,1042,2024-05-14 10:12:09,NaT
2,admin_auto_option_info,BASE TABLE,InnoDB,9,2024-05-14 10:12:09,NaT
3,agent_chat_file_info,BASE TABLE,InnoDB,98,2026-03-27 11:09:48,2026-06-05 13:40:31
4,agent_chat_info,BASE TABLE,InnoDB,174,2026-03-25 14:59:04,2026-06-12 16:25:50
...,...,...,...,...,...,...
130,user_log,BASE TABLE,InnoDB,255839,2024-05-14 10:14:34,2026-06-18 11:11:32
131,weather_data,BASE TABLE,InnoDB,494020,2024-11-08 09:15:31,2026-06-18 14:40:32
132,weather_fcst_info,BASE TABLE,InnoDB,3890,2025-04-22 18:29:52,NaT
133,weather_observ_info,BASE TABLE,InnoDB,211,2024-05-14 10:14:38,NaT


## 4. Inspect One Table

In [45]:
# TABLE/WHERE/SAMPLE_SIZE 값을 바꾸면 원하는 테이블과 조건으로 다시 실행할 수 있습니다.
inspect_table(
    conn,
    schema=cfg.name,
    table=TABLE,
    sample_size=SAMPLE_SIZE,
    order_by=ORDER_BY,
    where=WHERE,
)


## Columns: farmstom.data_silla_enc


,pos,column_name,column_type,nullable,column_key,default_value,extra
0,1,idx,int(11),NO,PRI,NaN,auto_increment
1,2,iot_data_idx,varchar(50),NO,MUL,NaN,
2,3,reg_date,datetime,YES,,NULL,
3,4,in_medium_temp1,varchar(10),YES,,NULL,
4,5,in_medium_temp2,varchar(10),YES,,NULL,
5,6,in_temp,varchar(10),YES,,NULL,
6,7,in_temp2,varchar(10),YES,,NULL,
7,8,in_water_hot,varchar(10),YES,,NULL,
8,9,in_water_cold,varchar(10),YES,,NULL,
9,10,in_hum,varchar(10),YES,,NULL,


## Sample Rows: farmstom.data_silla_enc


,idx,iot_data_idx,reg_date,in_medium_temp1,in_medium_temp2,in_temp,in_temp2,in_water_hot,in_water_cold,in_hum,...,cont_cur_vol,cont_kwcur_vol,cont_heater_run,cont_cooler_run,cont_co2_run,cont_3way1_vol,cont_3way2_vol,cont_pump1_run,cont_pump2_run,cont_fan_run
0,1170248,97,2026-06-18 14:40:00,33.9,34,41.6,41.2,39.74,33.99,34.57,...,100,0,1,0,0,86,0,1,0,1
1,1170242,97,2026-06-18 14:35:00,33.9,34,41.5,41.2,39.02,33.99,34.4,...,100,0,1,0,0,86,0,1,0,1
2,1170236,97,2026-06-18 14:30:00,33.9,34,41.5,41.1,39.02,33.99,35.58,...,100,0,1,0,0,86,0,1,0,1
3,1170230,97,2026-06-18 14:25:00,33.9,34,41.5,41.1,39.02,33.99,35.58,...,100,0,1,0,0,86,0,1,0,1
4,1170224,97,2026-06-18 14:20:00,33.9,34,41.4,41.1,39.02,33.99,36.63,...,100,0,1,0,0,86,0,1,0,1


## 5. Optional: Inspect Every Table

This can print a lot of output. Run only when needed.

In [46]:
# # 전체 테이블을 한 번에 보고 싶으면 아래 주석을 해제해서 실행합니다.
# for row in tables:
#     inspect_table(
#         conn,
#         schema=cfg.name,
#         table=row["table_name"],
#         sample_size=3,
#         order_by=ORDER_BY,
#         where=None,
#     )


## 6. Scan Non-Zero Values by Column and Period

지정한 컬럼마다 0이 아닌 값이 존재하는지 확인하고, 전체 및 기간별 개수와 비율을 집계합니다.

In [47]:
# 검사할 컬럼 목록입니다. 요청한 컬럼 순서를 그대로 유지합니다.
TARGET_COLUMNS = [
    "in_medium_temp1",
    "in_medium_temp2",
    "in_medium_hum2",
    "in_medium_ec2",
    "out_rainfall",
    "out_rain",
    "etc_blackout",
    "etc_plc_abnorm",
    "etc_plc_norm",
    "cont_skyl_vol",
    "cont_skyr_vol",
    "cont_kwcur_vol",
    "cont_heater_run",
    "cont_cooler_run",
    "cont_co2_run",
    "cont_3way1_vol",
    "cont_3way2_vol",
    "cont_pump1_run",
    "cont_pump2_run",
    "cont_fan_run",
]

SCAN_TABLE = TABLE
SCAN_TIME_COLUMN = "reg_date"

# 기간 구분 단위: "day", "month", "year" 중 하나를 선택합니다.
PERIOD_UNIT = "month"

# None이면 전체 기간을 검사합니다. 종료일은 해당 시각 미만(<) 조건입니다.
START_DATE = None  # 예: "2025-01-01 00:00:00"
END_DATE = None    # 예: "2026-01-01 00:00:00"

# 전체 농장을 검사하려면 None을 유지합니다.
# 특정 농장만 검사하려면 예: SCAN_EXTRA_WHERE = "iot_data_idx = 97"
SCAN_EXTRA_WHERE = None


In [48]:
def build_scan_filter(
    time_column: str,
    start_date: str | None,
    end_date: str | None,
    extra_where: str | None,
) -> tuple[str, list[Any]]:
    # 날짜 범위는 파라미터 바인딩을 사용하고, 추가 조건은 선택적으로 붙입니다.
    clauses = []
    params: list[Any] = []
    quoted_time = quote_identifier(time_column)

    if start_date:
        clauses.append(f"{quoted_time} >= %s")
        params.append(start_date)
    if end_date:
        clauses.append(f"{quoted_time} < %s")
        params.append(end_date)
    if extra_where:
        clauses.append(f"({extra_where})")

    where_sql = " WHERE " + " AND ".join(clauses) if clauses else ""
    return where_sql, params


def scan_nonzero_columns(
    conn,
    *,
    schema: str,
    table: str,
    target_columns: Sequence[str],
    time_column: str = "reg_date",
    period_unit: str = "month",
    start_date: str | None = None,
    end_date: str | None = None,
    extra_where: str | None = None,
) -> dict[str, Any]:
    # 실제 DB에 존재하는 컬럼만 조회하여 컬럼명 차이로 인한 SQL 오류를 막습니다.
    metadata = fetch_columns(conn, schema, table)
    existing_columns = {row["column_name"] for row in metadata}
    available = [name for name in target_columns if name in existing_columns]
    missing = [name for name in target_columns if name not in existing_columns]

    if not available:
        raise ValueError("검사 대상 컬럼이 테이블에 하나도 없습니다.")
    if time_column not in existing_columns:
        raise ValueError(f"기간 구분 컬럼 '{time_column}'이 테이블에 없습니다.")

    period_expressions = {
        "day": f"DATE({quote_identifier(time_column)})",
        # PyMySQL의 % 파라미터 처리와 충돌하지 않도록 %%로 이스케이프합니다.
        "month": f"DATE_FORMAT({quote_identifier(time_column)}, '%%Y-%%m')",
        "year": f"YEAR({quote_identifier(time_column)})",
    }
    if period_unit not in period_expressions:
        raise ValueError("period_unit은 'day', 'month', 'year' 중 하나여야 합니다.")

    where_sql, params = build_scan_filter(
        time_column,
        start_date,
        end_date,
        extra_where,
    )

    # 각 컬럼의 0이 아닌 행 개수를 DB 서버에서 직접 계산합니다.
    count_expressions = [
        "SUM(CASE WHEN COALESCE({column}, 0) <> 0 THEN 1 ELSE 0 END) AS {alias}".format(
            column=quote_identifier(column),
            alias=quote_identifier(column),
        )
        for column in available
    ]
    quoted_table = quote_identifier(table)

    overall_sql = (
        "SELECT COUNT(*) AS total_rows,\n       "
        + ",\n       ".join(count_expressions)
        + f"\nFROM {quoted_table}{where_sql}"
    )
    with conn.cursor() as cur:
        cur.execute(overall_sql, tuple(params))
        overall_raw = cur.fetchone()

    total_rows = int(overall_raw["total_rows"] or 0)
    overall_rows = []
    for column in available:
        nonzero_count = int(overall_raw[column] or 0)
        overall_rows.append(
            {
                "column_name": column,
                "nonzero_exists": "있음" if nonzero_count > 0 else "없음",
                "nonzero_count": nonzero_count,
                "total_rows": total_rows,
                "nonzero_ratio_pct": round(nonzero_count / total_rows * 100, 4) if total_rows else 0.0,
            }
        )

    # reg_date가 NULL인 행은 기간별 그룹에 넣을 수 없으므로 기간 집계에서 제외합니다.
    time_not_null = f"{quote_identifier(time_column)} IS NOT NULL"
    period_where_sql = (
        f"{where_sql} AND {time_not_null}" if where_sql else f" WHERE {time_not_null}"
    )
    period_expression = period_expressions[period_unit]
    period_sql = (
        f"SELECT {period_expression} AS period, COUNT(*) AS period_rows,\n       "
        + ",\n       ".join(count_expressions)
        + f"\nFROM {quoted_table}{period_where_sql}"
        + f"\nGROUP BY {period_expression} ORDER BY {period_expression}"
    )
    with conn.cursor() as cur:
        cur.execute(period_sql, tuple(params))
        grouped_rows = list(cur.fetchall())

    period_rows = []
    for grouped in grouped_rows:
        period_total = int(grouped["period_rows"] or 0)
        for column in available:
            nonzero_count = int(grouped[column] or 0)
            period_rows.append(
                {
                    "period": grouped["period"],
                    "column_name": column,
                    "nonzero_count": nonzero_count,
                    "period_rows": period_total,
                    "nonzero_ratio_pct": round(nonzero_count / period_total * 100, 4) if period_total else 0.0,
                }
            )

    return {
        "overall": overall_rows,
        "period": period_rows,
        "missing_columns": missing,
    }


In [35]:
# 연결이 오래되어 끊겼다면 같은 설정으로 자동 재접속합니다.
try:
    conn.ping(reconnect=True)
except Exception:
    conn = connect(cfg)

scan_result = scan_nonzero_columns(
    conn,
    schema=cfg.name,
    table=SCAN_TABLE,
    target_columns=TARGET_COLUMNS,
    time_column=SCAN_TIME_COLUMN,
    period_unit=PERIOD_UNIT,
    start_date=START_DATE,
    end_date=END_DATE,
    extra_where=SCAN_EXTRA_WHERE,
)

if scan_result["missing_columns"]:
    print("[테이블에 없는 컬럼]")
    print(", ".join(scan_result["missing_columns"]))
    print()

print("[전체 기간: 컬럼별 0이 아닌 값 집계]")
show_table(scan_result["overall"])

print(f"[{PERIOD_UNIT} 단위: 0이 아닌 값 개수]")
if pd is not None and scan_result["period"]:
    period_df = pd.DataFrame(scan_result["period"])
    count_pivot = period_df.pivot(
        index="period",
        columns="column_name",
        values="nonzero_count",
    ).fillna(0).astype(int)
    display(count_pivot)

    print(f"[{PERIOD_UNIT} 단위: 0이 아닌 값 비율(%)]")
    ratio_pivot = period_df.pivot(
        index="period",
        columns="column_name",
        values="nonzero_ratio_pct",
    ).fillna(0.0)
    display(ratio_pivot)
else:
    show_table(scan_result["period"])


C:\Users\10-64\AppData\Local\Temp\ipykernel_14776\2403996858.py:3: DeprecationWarning: The 'reconnect' argument is deprecated. Create a new connection if you want to reconnect.
  conn.ping(reconnect=True)


[전체 기간: 컬럼별 0이 아닌 값 집계]


,column_name,nonzero_exists,nonzero_count,total_rows,nonzero_ratio_pct
0,in_medium_temp1,있음,1160131,1160573,99.9619
1,in_medium_temp2,있음,550786,1160573,47.4581
2,in_medium_hum2,있음,550845,1160573,47.4632
3,in_medium_ec2,있음,221932,1160573,19.1226
4,out_rainfall,있음,62021,1160573,5.3440
5,out_rain,있음,85862,1160573,7.3982
6,etc_blackout,없음,0,1160573,0.0000
7,etc_plc_abnorm,없음,0,1160573,0.0000
8,etc_plc_norm,있음,620716,1160573,53.4836
9,cont_skyl_vol,있음,530620,1160573,45.7205


[month 단위: 0이 아닌 값 개수]


column_name,cont_3way1_vol,cont_3way2_vol,cont_co2_run,cont_cooler_run,cont_fan_run,cont_heater_run,cont_kwcur_vol,cont_pump1_run,cont_pump2_run,cont_skyl_vol,cont_skyr_vol,etc_blackout,etc_plc_abnorm,etc_plc_norm,in_medium_ec2,in_medium_hum2,in_medium_temp1,in_medium_temp2,out_rain,out_rainfall
period,,,,,,,,,,,,,,,,,,,,
2024-03,1170,1158,0,1084,4164,2116,5498,2133,1949,8059,8056,0,0,6192,0,0,10948,0,0,448
2024-04,588,3477,0,3444,7374,3444,10896,564,3444,9675,9675,0,0,6828,0,0,12786,0,954,498
2024-05,0,9455,0,9437,9726,9437,13621,0,9437,10851,10851,0,0,7256,366,366,13731,153,1275,1056
2024-06,0,9561,0,9533,13697,9533,17248,0,9533,13507,13507,0,0,9986,0,0,17252,0,1689,536
2024-07,0,19341,0,19341,19341,11277,24899,0,19341,13184,13184,0,0,14023,0,0,24899,0,6541,1961
2024-08,1860,22056,0,22056,8511,22056,26568,1860,22056,25164,24990,0,0,11997,3642,3636,26580,3642,1470,252
2024-09,6,33624,0,32316,16498,8736,33172,6,32400,27408,27186,0,0,25002,234,222,43462,240,5670,2046
2024-10,516,504,0,396,13482,360,51126,480,390,36420,36402,0,0,22935,6072,6186,51828,6150,6762,8034
2024-11,1116,606,0,528,11490,528,36960,1008,528,25944,25944,0,0,23814,6,12,37848,12,1746,288


[month 단위: 0이 아닌 값 비율(%)]


column_name,cont_3way1_vol,cont_3way2_vol,cont_co2_run,cont_cooler_run,cont_fan_run,cont_heater_run,cont_kwcur_vol,cont_pump1_run,cont_pump2_run,cont_skyl_vol,cont_skyr_vol,etc_blackout,etc_plc_abnorm,etc_plc_norm,in_medium_ec2,in_medium_hum2,in_medium_temp1,in_medium_temp2,out_rain,out_rainfall
period,,,,,,,,,,,,,,,,,,,,
2024-03,10.6869,10.5773,0.0,9.9014,38.0343,19.3277,50.2192,19.4830,17.8023,73.6116,73.5842,0.0,0.0,56.5583,0.0000,0.0000,100.0000,0.0000,0.0000,4.0921
2024-04,4.5988,27.1938,0.0,26.9357,57.6725,26.9357,85.2182,4.4111,26.9357,75.6687,75.6687,0.0,0.0,53.4022,0.0000,0.0000,100.0000,0.0000,7.4613,3.8949
2024-05,0.0000,68.8588,0.0,68.7277,70.8324,68.7277,99.1989,0.0000,68.7277,79.0256,79.0256,0.0,0.0,52.8439,2.6655,2.6655,100.0000,1.1143,9.2856,7.6906
2024-06,0.0000,55.4197,0.0,55.2574,79.3937,55.2574,99.9768,0.0000,55.2574,78.2924,78.2924,0.0,0.0,57.8831,0.0000,0.0000,100.0000,0.0000,9.7902,3.1069
2024-07,0.0000,77.6778,0.0,77.6778,77.6778,45.2910,100.0000,0.0000,77.6778,52.9499,52.9499,0.0,0.0,56.3195,0.0000,0.0000,100.0000,0.0000,26.2701,7.8758
2024-08,6.9977,82.9797,0.0,82.9797,32.0203,82.9797,99.9549,6.9977,82.9797,94.6727,94.0181,0.0,0.0,45.1354,13.7020,13.6795,100.0000,13.7020,5.5305,0.9481
2024-09,0.0138,77.3641,0.0,74.3546,37.9596,20.1003,76.3241,0.0138,74.5479,63.0620,62.5512,0.0,0.0,57.5261,0.5384,0.5108,100.0000,0.5522,13.0459,4.7076
2024-10,0.9956,0.9724,0.0,0.7641,26.0130,0.6946,98.6455,0.9261,0.7525,70.2709,70.2362,0.0,0.0,44.2521,11.7157,11.9356,100.0000,11.8662,13.0470,15.5013
2024-11,2.9486,1.6011,0.0,1.3951,30.3583,1.3951,97.6538,2.6633,1.3951,68.5479,68.5479,0.0,0.0,62.9201,0.0159,0.0317,100.0000,0.0317,4.6132,0.7609


## 7. Summary Statistics for All Columns

지정한 41개 컬럼을 대상으로 최솟값, 최댓값, 평균, 중앙값을 정리합니다. 숫자처럼 저장된 문자열도 숫자로 변환해 계산합니다.

In [36]:
# 결과에 표시할 41개 컬럼의 순서를 명시적으로 고정합니다.
STAT_COLUMN_ORDER = [
    "idx",
    "iot_data_idx",
    "reg_date",
    "in_medium_temp1",
    "in_medium_temp2",
    "in_temp",
    "in_temp2",
    "in_water_hot",
    "in_water_cold",
    "in_hum",
    "in_hum2",
    "in_medium_hum1",
    "in_medium_hum2",
    "in_co2",
    "in_co2_2",
    "in_medium_ec1",
    "in_medium_ec2",
    "out_temp",
    "out_hum",
    "out_winddirec",
    "out_windsp",
    "out_light",
    "out_light_sum",
    "out_rainfall",
    "out_rain",
    "out_airpress",
    "etc_blackout",
    "etc_plc_abnorm",
    "etc_plc_norm",
    "cont_skyl_vol",
    "cont_skyr_vol",
    "cont_cur_vol",
    "cont_kwcur_vol",
    "cont_heater_run",
    "cont_cooler_run",
    "cont_co2_run",
    "cont_3way1_vol",
    "cont_3way2_vol",
    "cont_pump1_run",
    "cont_pump2_run",
    "cont_fan_run",
]


def numeric_sql_expression(column: str) -> str:
    # VARCHAR로 저장된 숫자도 숫자 기준으로 MIN/MAX/AVG가 계산되게 변환합니다.
    # 숫자가 아닌 문자열과 빈 문자열은 NULL로 처리하여 통계에서 제외합니다.
    quoted = quote_identifier(column)
    return (
        "CASE WHEN TRIM(CAST({column} AS CHAR)) "
        "REGEXP '^[+-]?([0-9]+([.][0-9]*)?|[.][0-9]+)([eE][+-]?[0-9]+)?$' "
        "THEN CAST({column} AS DECIMAL(65, 20)) ELSE NULL END"
    ).format(column=quoted)


def calculate_exact_median(
    conn,
    *,
    table: str,
    column: str,
    numeric_count: int,
    where_sql: str,
    where_params: Sequence[Any],
) -> float | None:
    # MySQL에는 버전에 상관없이 쓸 수 있는 MEDIAN 함수가 없으므로,
    # 정렬된 값의 가운데 1개(홀수) 또는 2개(짝수)를 가져와 평균합니다.
    if numeric_count <= 0:
        return None

    if numeric_count % 2 == 1:
        offset = numeric_count // 2
        limit = 1
    else:
        offset = numeric_count // 2 - 1
        limit = 2

    numeric_expression = numeric_sql_expression(column)
    nonnull_clause = f"({numeric_expression}) IS NOT NULL"
    median_where = (
        f"{where_sql} AND {nonnull_clause}"
        if where_sql
        else f" WHERE {nonnull_clause}"
    )
    sql = f"""
        SELECT AVG(median_value) AS median_value
        FROM (
            SELECT {numeric_expression} AS median_value
            FROM {quote_identifier(table)}
            {median_where}
            ORDER BY median_value
            LIMIT {offset}, {limit}
        ) AS median_rows
    """
    with conn.cursor() as cur:
        cur.execute(sql, tuple(where_params))
        value = cur.fetchone()["median_value"]
    return float(value) if value is not None else None


def summarize_all_columns(
    conn,
    *,
    schema: str,
    table: str,
    time_column: str = "reg_date",
    start_date: str | None = None,
    end_date: str | None = None,
    extra_where: str | None = None,
    include_median: bool = True,
) -> list[dict[str, Any]]:
    # DB 메타데이터는 컬럼 존재 여부 확인에만 사용하고 결과 순서는 위 목록을 따릅니다.
    metadata_rows = fetch_columns(conn, schema, table)
    if not metadata_rows:
        raise ValueError(f"테이블 '{schema}.{table}'의 컬럼 정보를 찾을 수 없습니다.")
    existing_columns = {row["column_name"] for row in metadata_rows}

    where_sql, where_params = build_scan_filter(
        time_column,
        start_date,
        end_date,
        extra_where,
    )

    # reg_date는 날짜 MIN/MAX만 계산하고, 나머지는 모두 숫자로 검증·변환해 집계합니다.
    expressions = []
    available_columns = [name for name in STAT_COLUMN_ORDER if name in existing_columns]
    for index, column in enumerate(available_columns):
        if column == "reg_date":
            quoted = quote_identifier(column)
            expressions.extend(
                [
                    f"MIN({quoted}) AS {quote_identifier(f'c{index}_min')}",
                    f"MAX({quoted}) AS {quote_identifier(f'c{index}_max')}",
                ]
            )
            continue

        numeric_expression = numeric_sql_expression(column)
        expressions.extend(
            [
                f"COUNT({numeric_expression}) AS {quote_identifier(f'c{index}_count')}",
                f"MIN({numeric_expression}) AS {quote_identifier(f'c{index}_min')}",
                f"MAX({numeric_expression}) AS {quote_identifier(f'c{index}_max')}",
                f"AVG({numeric_expression}) AS {quote_identifier(f'c{index}_mean')}",
            ]
        )

    sql = (
        "SELECT\n    "
        + ",\n    ".join(expressions)
        + f"\nFROM {quote_identifier(table)}{where_sql}"
    )
    with conn.cursor() as cur:
        cur.execute(sql, tuple(where_params))
        aggregate = cur.fetchone()

    result = []
    available_index = {name: index for index, name in enumerate(available_columns)}
    for column in STAT_COLUMN_ORDER:
        # DB에 없는 컬럼도 결과 순서를 유지하며 '-'로 표시합니다.
        if column not in available_index:
            result.append(
                {"column_name": column, "min": "-", "max": "-", "mean": "-", "median": "-"}
            )
            continue

        index = available_index[column]
        if column == "reg_date":
            result.append(
                {
                    "column_name": column,
                    "min": aggregate[f"c{index}_min"] or "-",
                    "max": aggregate[f"c{index}_max"] or "-",
                    "mean": "-",
                    "median": "-",
                }
            )
            continue

        numeric_count = int(aggregate[f"c{index}_count"] or 0)
        min_value = aggregate[f"c{index}_min"]
        max_value = aggregate[f"c{index}_max"]
        mean_value = aggregate[f"c{index}_mean"]
        median_value = None
        if include_median and numeric_count > 0:
            # 정확 중앙값 계산은 숫자형 컬럼마다 정렬 쿼리를 실행하므로 시간이 걸릴 수 있습니다.
            median_value = calculate_exact_median(
                conn,
                table=table,
                column=column,
                numeric_count=numeric_count,
                where_sql=where_sql,
                where_params=where_params,
            )

        result.append(
            {
                "column_name": column,
                "min": float(min_value) if min_value is not None else "-",
                "max": float(max_value) if max_value is not None else "-",
                "mean": float(mean_value) if mean_value is not None else "-",
                "median": median_value if median_value is not None else "-",
            }
        )

    return result


In [37]:
# 통계 계산 범위입니다. None이면 DB 전체 기간을 대상으로 합니다.
STATS_START_DATE = None
STATS_END_DATE = None
STATS_EXTRA_WHERE = None  # 예: "iot_data_idx = 97"

# True이면 정확 중앙값을 계산합니다. 전체 DB에서는 시간이 오래 걸릴 수 있습니다.
INCLUDE_MEDIAN = True

try:
    conn.ping(reconnect=True)
except Exception:
    conn = connect(cfg)

column_statistics = summarize_all_columns(
    conn,
    schema=cfg.name,
    table=TABLE,
    time_column="reg_date",
    start_date=STATS_START_DATE,
    end_date=STATS_END_DATE,
    extra_where=STATS_EXTRA_WHERE,
    include_median=INCLUDE_MEDIAN,
)

print(f"[컬럼 통계] 총 {len(column_statistics)}개 컬럼")
if pd is not None:
    statistics_df = pd.DataFrame(column_statistics).fillna("-")
    statistics_df = statistics_df[["column_name", "min", "max", "mean", "median"]]
    display(statistics_df)
else:
    show_table(column_statistics)


C:\Users\10-64\AppData\Local\Temp\ipykernel_14776\1896604722.py:10: DeprecationWarning: The 'reconnect' argument is deprecated. Create a new connection if you want to reconnect.
  conn.ping(reconnect=True)


[컬럼 통계] 총 41개 컬럼


,column_name,min,max,mean,median
0,idx,1.0,1170247.0,589296.133198,589961.0
1,iot_data_idx,97.0,281.0,131.782418,99.0
2,reg_date,2024-03-05 10:50:00,2026-06-18 14:35:00,-,-
3,in_medium_temp1,0.0,57.1,20.541599,19.07
4,in_medium_temp2,0.0,55.85,11.1049,0.0
5,in_temp,2.3,48.3,22.638696,22.13
6,in_temp2,2.3,48.3,22.312552,21.65
7,in_water_hot,-47.19,470.24,32.503068,28.0
8,in_water_cold,-19.38,475.68,23.064884,20.01
9,in_hum,7.8,83.85,47.402928,47.18


## 8. Production Values in Separate Tables by Date

`grow_result`에서 생산 관련 값이 하나라도 0이 아닌 행을 조회하고, 측정 날짜마다 독립적인 표로 출력합니다.

In [38]:
# 날짜별로 확인할 생산 관련 컬럼입니다.
PRODUCTION_VALUE_COLUMNS = [
    "productionUnit",   # 생산단수(제곱미터당 수확량)
    "harvOverWgh",      # 수확과중
    "harvCnt",          # 수확과수
    "nonproductWgh",    # 비상품과중
    "nonproductCnt",    # 비상품과수
]


def fetch_production_rows_by_date(
    conn,
    *,
    table: str = "grow_result",
    date_column: str = "msrDate",
    value_columns: Sequence[str] = PRODUCTION_VALUE_COLUMNS,
    start_date: str | None = None,
    end_date: str | None = None,
) -> list[dict[str, Any]]:
    # 기존 노트북의 주석 처리된 코드는 건드리지 않고 별도의 조회문을 사용합니다.
    selected_columns = ["grow_idx", "cropIdx", date_column, *value_columns]
    select_sql = ", ".join(quote_identifier(name) for name in selected_columns)

    # 생산 관련 컬럼 중 하나라도 0이 아니면 조회 대상에 포함합니다.
    nonzero_conditions = [
        f"COALESCE({quote_identifier(name)}, 0) <> 0"
        for name in value_columns
    ]
    clauses = ["(" + " OR ".join(nonzero_conditions) + ")"]
    params: list[Any] = []

    if start_date:
        clauses.append(f"{quote_identifier(date_column)} >= %s")
        params.append(start_date)
    if end_date:
        clauses.append(f"{quote_identifier(date_column)} < %s")
        params.append(end_date)

    sql = f"""
        SELECT {select_sql}
        FROM {quote_identifier(table)}
        WHERE {' AND '.join(clauses)}
        ORDER BY {quote_identifier(date_column)}, {quote_identifier('grow_idx')}
    """
    with conn.cursor() as cur:
        cur.execute(sql, tuple(params))
        return list(cur.fetchall())


def display_production_tables_by_date(
    rows: list[dict[str, Any]],
    *,
    date_column: str = "msrDate",
) -> None:
    if not rows:
        print("생산 관련 값이 0이 아닌 행이 없습니다.")
        return

    # 날짜 부분만 기준으로 묶어 같은 날짜의 여러 행을 하나의 표에 표시합니다.
    rows_by_date: dict[str, list[dict[str, Any]]] = {}
    for row in rows:
        raw_date = row.get(date_column)
        date_key = raw_date.date().isoformat() if hasattr(raw_date, "date") else str(raw_date)[:10]
        rows_by_date.setdefault(date_key, []).append(row)

    for date_key, date_rows in rows_by_date.items():
        print(f"\n### {date_key}")
        if pd is not None:
            date_df = pd.DataFrame(date_rows).fillna("-")
            display(date_df)
        else:
            show_table(date_rows)


In [39]:
# None이면 전체 기간을 조회합니다. 필요한 경우 날짜 범위를 지정할 수 있습니다.
PRODUCTION_START_DATE = None  # 예: "2024-01-01 00:00:00"
PRODUCTION_END_DATE = None    # 예: "2025-01-01 00:00:00"

try:
    conn.ping(reconnect=True)
except Exception:
    conn = connect(cfg)

production_rows = fetch_production_rows_by_date(
    conn,
    start_date=PRODUCTION_START_DATE,
    end_date=PRODUCTION_END_DATE,
)

print(f"[생산 관련 행] 총 {len(production_rows)}개")
display_production_tables_by_date(production_rows)


[생산 관련 행] 총 11개

### 2023-12-27


C:\Users\10-64\AppData\Local\Temp\ipykernel_14776\3995504013.py:6: DeprecationWarning: The 'reconnect' argument is deprecated. Create a new connection if you want to reconnect.
  conn.ping(reconnect=True)


,grow_idx,cropIdx,msrDate,productionUnit,harvOverWgh,harvCnt,nonproductWgh,nonproductCnt
0,8,1,2023-12-27,5.0,-,-,-,-



### 2024-01-02


,grow_idx,cropIdx,msrDate,productionUnit,harvOverWgh,harvCnt,nonproductWgh,nonproductCnt
0,85,1,2024-01-02,11.0,-,-,-,-


...

,grow_idx,cropIdx,msrDate,productionUnit,harvOverWgh,harvCnt,nonproductWgh,nonproductCnt
0,357,20,2024-08-13,15.0,-,-,-,-



### 2024-10-31


,grow_idx,cropIdx,msrDate,productionUnit,harvOverWgh,harvCnt,nonproductWgh,nonproductCnt
0,1035,35,2024-10-31,4.0,0.0,0,0.0,0



### 2024-11-01


,grow_idx,cropIdx,msrDate,productionUnit,harvOverWgh,harvCnt,nonproductWgh,nonproductCnt
0,1037,35,2024-11-01,11.0,-,-,-,-


## 9. Close Connection

In [40]:
# 노트북 작업이 끝나면 DB 연결을 닫습니다.
conn.close()
